In [2]:

# ============================================================
# MODULE 4: OLIST REPEAT-PURCHASE PREDICTION
# STEP 1: ENVIRONMENT SETUP AND DATA INSPECTION
# ============================================================

# 1. Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# 2. Define project and dataset paths
PROJECT_DIR = Path(
    r"C:\Users\HP\PycharmProjects\Learning\Nex_Project"
)

DELIVERED_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "delivered_order_level.csv"
)

# 3. Load the Module 3 delivered-order dataset
df = pd.read_csv(DELIVERED_FILE)

# 4. Convert relevant date columns to datetime
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(
            df[column],
            errors="coerce"
        )

# 5. Inspect the dataset
print("DATASET OVERVIEW")
print("-" * 50)

print("Dataset shape:", df.shape)

print(
    "Unique customers:",
    df["customer_unique_id"].nunique()
)

print(
    "Unique orders:",
    df["order_id"].nunique()
)

print("\nCOLUMN NAMES")
print(df.columns.tolist())

print("\nDATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
print(
    df.isnull().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nPURCHASE DATE RANGE")
print("Earliest:", df["order_purchase_timestamp"].min())
print("Latest:", df["order_purchase_timestamp"].max())

print("\nSAMPLE RECORDS")
display(df.head())


DATASET OVERVIEW
--------------------------------------------------
Dataset shape: (96478, 31)
Unique customers: 93358
Unique orders: 96478

COLUMN NAMES
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_spending', 'freight_spending', 'total_items', 'distinct_products', 'category_diversity', 'total_payment_value', 'payment_record_count', 'preferred_payment_type', 'average_review_score', 'review_record_count', 'approval_date_missing', 'carrier_date_missing', 'delivery_date_missing', 'approval_before_purchase', 'carrier_before_approval', 'delivery_before_carrier', 'delivery_after_estimate', 'carrier_approval_gap_hours', 'delivery_carrier_gap_hours']

DATA TYPES
order_id                                    str
customer_id                                 str
orde

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,review_record_count,approval_date_missing,carrier_date_missing,delivery_date_missing,approval_before_purchase,carrier_before_approval,delivery_before_carrier,delivery_after_estimate,carrier_approval_gap_hours,delivery_carrier_gap_hours
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,False,False,False,False,False,False,False,56.795833,145.503611
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,False,False,False,False,False,False,False,11.109167,288.945833
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,False,False,False,False,False,False,False,4.910278,220.274722
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,1.0,False,False,False,False,False,False,False,89.900000,226.811944
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,1.0,False,False,False,False,False,False,False,21.434722,46.507778


In [ ]:

# ============================================================
# STEP 2: DEFINE THE REPEAT-PURCHASE PREDICTION TARGET
# ============================================================

# 1. Define the observation and prediction periods
LOOKBACK_DAYS = 180
PREDICTION_DAYS = 90

# Use the latest purchase date to determine the prediction cutoff.
# This ensures that the future window is covered by the dataset.
latest_purchase_date = df["order_purchase_timestamp"].max()

prediction_date = (
    latest_purchase_date
    - pd.Timedelta(days=PREDICTION_DAYS)
)

observation_start = (
    prediction_date
    - pd.Timedelta(days=LOOKBACK_DAYS)
)

prediction_end = prediction_date + pd.Timedelta(
    days=PREDICTION_DAYS
)

print("PREDICTION WINDOW")
print("-" * 50)
print("Observation starts:", observation_start)
print("Prediction date:", prediction_date)
print("Prediction ends:", prediction_end)

# 2. Separate historical purchases from future purchases.
# Historical purchases are used to create customer features.
# Future purchases are used ONLY to create the target.

historical_orders = df[
    (df["order_purchase_timestamp"] >= observation_start) &
    (df["order_purchase_timestamp"] < prediction_date)
].copy()

future_orders = df[
    (df["order_purchase_timestamp"] >= prediction_date) &
    (df["order_purchase_timestamp"] <= prediction_end)
].copy()

print("\nORDER COUNTS")
print("-" * 50)
print("Historical orders:", len(historical_orders))
print("Future orders:", len(future_orders))

# 3. Identify customers eligible for prediction.
# A customer must have at least one historical delivered purchase.

eligible_customers = (
    historical_orders["customer_unique_id"]
    .dropna()
    .unique()
)

print("\nELIGIBLE CUSTOMERS:", len(eligible_customers))

# 4. Count future purchases for each eligible customer.
future_purchase_counts = (
    future_orders[
        future_orders["customer_unique_id"].isin(eligible_customers)
    ]
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

# 5. Create the customer-level target.
# 1 = At least one future delivered purchase.
# 0 = No future delivered purchase within the prediction window.

target = pd.DataFrame({
    "customer_unique_id": eligible_customers
})

target["future_order_count"] = (
    target["customer_unique_id"]
    .map(future_purchase_counts)
    .fillna(0)
    .astype(int)
)

target["repeat_purchase"] = (
    target["future_order_count"] > 0
).astype(int)

# 6. Inspect the target distribution.
target_distribution = (
    target["repeat_purchase"]
    .value_counts()
    .sort_index()
    .rename(index={
        0: "No repeat purchase",
        1: "Repeat purchase"
    })
)

target_percentages = (
    target["repeat_purchase"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .rename(index={
        0: "No repeat purchase",
        1: "Repeat purchase"
    })
)

print("\nTARGET DISTRIBUTION")
print("-" * 50)
print(pd.DataFrame({
    "Customers": target_distribution,
    "Percentage": target_percentages.round(2)
}))

# 7. Visualize the target distribution.
plt.figure(figsize=(7, 5))

sns.countplot(
    data=target,
    x="repeat_purchase"
)

plt.title("Customer Repeat-Purchase Distribution")
plt.xlabel("Repeat Purchase (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")
plt.xticks([0, 1], ["No Repeat", "Repeat"])
plt.tight_layout()
plt.show()

# 8. Basic validation checks.
assert target["customer_unique_id"].is_unique
assert target["repeat_purchase"].notna().all()
assert set(target["repeat_purchase"].unique()).issubset({0, 1})

print("\nTARGET VALIDATION: PASSED")
print("Unique customer records:", target["customer_unique_id"].nunique())


In [ ]:

# ============================================================
# STEP 3: COMPARE REPEAT-PURCHASE PREDICTION HORIZONS
# ============================================================

LOOKBACK_DAYS = 180
HORIZONS = [90, 120, 180]

latest_purchase_date = df["order_purchase_timestamp"].max()

horizon_results = []

for horizon in HORIZONS:

    # Set prediction date so the complete future window
    # is covered by the available dataset.
    cutoff_date = (
        latest_purchase_date
        - pd.Timedelta(days=horizon)
    )

    observation_start = (
        cutoff_date
        - pd.Timedelta(days=LOOKBACK_DAYS)
    )

    # Historical customer population
    historical = df[
        (df["order_purchase_timestamp"] >= observation_start) &
        (df["order_purchase_timestamp"] < cutoff_date)
    ].copy()

    eligible = historical["customer_unique_id"].dropna().unique()

    # Future delivered purchases
    future = df[
        (df["order_purchase_timestamp"] >= cutoff_date) &
        (df["order_purchase_timestamp"] <= latest_purchase_date)
    ].copy()

    future_counts = (
        future[
            future["customer_unique_id"].isin(eligible)
        ]
        .groupby("customer_unique_id")["order_id"]
        .nunique()
    )

    repeat_customers = future_counts[
        future_counts > 0
    ].index.nunique()

    eligible_count = len(eligible)
    non_repeat_customers = eligible_count - repeat_customers

    repeat_rate = (
        repeat_customers / eligible_count * 100
        if eligible_count > 0 else 0
    )

    horizon_results.append({
        "Prediction Horizon (Days)": horizon,
        "Prediction Date": cutoff_date,
        "Observation Start": observation_start,
        "Eligible Customers": eligible_count,
        "Repeat Customers": repeat_customers,
        "Non-Repeat Customers": non_repeat_customers,
        "Repeat Rate (%)": round(repeat_rate, 2)
    })

# Display comparison
horizon_comparison = pd.DataFrame(horizon_results)

display(horizon_comparison)

# Visualize repeat-purchase rates
plt.figure(figsize=(8, 5))

sns.barplot(
    data=horizon_comparison,
    x="Prediction Horizon (Days)",
    y="Repeat Rate (%)"
)

plt.title("Repeat-Purchase Rate by Prediction Horizon")
plt.xlabel("Prediction Horizon (Days)")
plt.ylabel("Repeat-Purchase Rate (%)")
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# INVESTIGATE CUSTOMER SATISFACTION PREDICTION
# ============================================================

# Review score distribution
review_distribution = (
    df["average_review_score"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Review Score")
    .reset_index(name="Number of Orders")
)

review_distribution["Percentage"] = (
    review_distribution["Number of Orders"]
    / len(df) * 100
).round(2)

display(review_distribution)

# Define low satisfaction as review scores of 1 or 2
df["low_satisfaction"] = np.where(
    df["average_review_score"].isna(),
    np.nan,
    (df["average_review_score"] <= 2).astype(int)
)

print("\nLOW SATISFACTION TARGET")
print("-" * 45)
print(df["low_satisfaction"].value_counts(dropna=False))

print("\nTARGET PERCENTAGES")
print(
    df["low_satisfaction"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

# Visualise review scores
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="average_review_score",
    order=[1, 2, 3, 4, 5]
)

plt.title("Olist Customer Review Score Distribution")
plt.xlabel("Review Score")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# PREPARE THE LOW-SATISFACTION MODELLING DATASET
# ============================================================

# 1. Keep only orders with an available review score
model_df = df.dropna(subset=["average_review_score"]).copy()

# 2. Create the binary classification target
# 1 = Low satisfaction (review score <= 2)
# 0 = Not low satisfaction (review score > 2)
model_df["low_satisfaction"] = (
    model_df["average_review_score"] <= 2
).astype(int)

# 3. Check dataset size
print("MODELLING DATASET")
print("Shape:", model_df.shape)
print("Unique orders:", model_df["order_id"].nunique())
print("Unique customers:", model_df["customer_unique_id"].nunique())

# 4. Check target distribution
target_distribution = (
    model_df["low_satisfaction"]
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Not Low Satisfaction",
        1: "Low Satisfaction"
    })
    .to_frame("Number of Orders")
)

target_distribution["Percentage"] = (
    target_distribution["Number of Orders"]
    / len(model_df) * 100
).round(2)

display(target_distribution)

# 5. Confirm there are no missing target values
assert model_df["low_satisfaction"].isna().sum() == 0

print("Target validation: PASSED")

In [ ]:

# ============================================================
# INSPECT CANDIDATE FEATURES FOR LOW-SATISFACTION PREDICTION
# ============================================================

# Candidate features that may be available at or shortly
# after order placement.
candidate_features = [
    "order_purchase_timestamp",
    "customer_state",
    "customer_zip_code_prefix",
    "item_spending",
    "freight_spending",
    "total_items",
    "distinct_products",
    "category_diversity",
    "total_payment_value",
    "payment_record_count",
    "preferred_payment_type"
]

# Check which candidate columns exist
available_features = [
    col for col in candidate_features
    if col in model_df.columns
]

# Create a feature inspection table
feature_check = pd.DataFrame({
    "Data Type": model_df[available_features].dtypes.astype(str),
    "Missing Values": model_df[available_features].isna().sum(),
    "Missing Percentage": (
        model_df[available_features].isna().mean() * 100
    ).round(2),
    "Unique Values": model_df[available_features].nunique()
})

display(feature_check)

print("\nCANDIDATE FEATURE SUMMARY")
print("Available candidate features:", len(available_features))
print(available_features)

# Inspect sample records
display(model_df[available_features + ["low_satisfaction"]].head())


In [ ]:

# ============================================================
# CREATE INITIAL MODELLING FEATURES
# ============================================================

# Work with a copy of the prepared modeling dataset
feature_df = model_df.copy()

# ------------------------------------------------------------
# 1. Extract time-based features from purchase timestamp
# ------------------------------------------------------------

feature_df["purchase_year"] = (
    feature_df["order_purchase_timestamp"].dt.year
)

feature_df["purchase_month"] = (
    feature_df["order_purchase_timestamp"].dt.month
)

feature_df["purchase_day_of_week"] = (
    feature_df["order_purchase_timestamp"].dt.dayofweek
)

feature_df["purchase_hour"] = (
    feature_df["order_purchase_timestamp"].dt.hour
)

# ------------------------------------------------------------
# 2. Select initial predictors
# ------------------------------------------------------------

# Payment record count is excluded because it may reflect
# payment processing after the prediction point.
# Total payment value is excluded initially until its timing
# and availability at checkout are confirmed.
# ZIP code is excluded initially due to high cardinality.

feature_columns = [
    "customer_state",
    "item_spending",
    "freight_spending",
    "total_items",
    "distinct_products",
    "category_diversity",
    "preferred_payment_type",
    "purchase_year",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour"
]

X = feature_df[feature_columns].copy()
y = feature_df["low_satisfaction"].copy()

# ------------------------------------------------------------
# 3. Check missing values
# ------------------------------------------------------------

print("FEATURE MATRIX")
print("Shape:", X.shape)

print("\nTARGET VECTOR")
print("Shape:", y.shape)

print("\nMISSING VALUES")
display(X.isnull().sum())

print("\nFEATURE DATA TYPES")
display(X.dtypes)

print("\nTARGET DISTRIBUTION")
display(y.value_counts(normalize=True).mul(100).round(2))

# ------------------------------------------------------------
# 4. Confirm no identifiers or target leakage fields
# ------------------------------------------------------------

assert "order_id" not in X.columns
assert "customer_id" not in X.columns
assert "customer_unique_id" not in X.columns
assert "average_review_score" not in X.columns
assert "low_satisfaction" not in X.columns
assert "order_delivered_customer_date" not in X.columns

print("\nInitial feature selection: PASSED")


In [ ]:

# ============================================================
# HANDLE MISSING VALUES AND CREATE CHRONOLOGICAL TRAIN/TEST SPLIT
# ============================================================

# 1. Replace the single missing payment type
X = X.copy()
X["preferred_payment_type"] = (
    X["preferred_payment_type"].fillna("Unknown")
)

# 2. Sort observations by purchase timestamp
# Keep the timestamp separately for splitting; it is not a predictor.
split_dates = feature_df.loc[X.index, "order_purchase_timestamp"]

sorted_indices = split_dates.sort_values().index

X_sorted = X.loc[sorted_indices].copy()
y_sorted = y.loc[sorted_indices].copy()
dates_sorted = split_dates.loc[sorted_indices].copy()

# 3. Use the 80th percentile purchase date as the cutoff
split_position = int(len(X_sorted) * 0.80)
cutoff_date = dates_sorted.iloc[split_position]

# 4. Create chronological training and testing sets
train_mask = dates_sorted < cutoff_date
test_mask = dates_sorted >= cutoff_date

X_train = X_sorted.loc[train_mask].copy()
X_test = X_sorted.loc[test_mask].copy()

y_train = y_sorted.loc[train_mask].copy()
y_test = y_sorted.loc[test_mask].copy()

# 5. Display split information
print("CHRONOLOGICAL DATA SPLIT")
print("Cutoff date:", cutoff_date)

print("\nTraining set")
print("Rows:", len(X_train))
print("Start:", dates_sorted.loc[train_mask].min())
print("End:", dates_sorted.loc[train_mask].max())

print("\nTesting set")
print("Rows:", len(X_test))
print("Start:", dates_sorted.loc[test_mask].min())
print("End:", dates_sorted.loc[test_mask].max())

# 6. Compare target distributions
split_summary = pd.DataFrame({
    "Training": y_train.value_counts(normalize=True),
    "Testing": y_test.value_counts(normalize=True)
}).fillna(0).mul(100).round(2)

split_summary.index = [
    "Not Low Satisfaction" if value == 0
    else "Low Satisfaction"
    for value in split_summary.index
]

print("\nTARGET DISTRIBUTION BY SPLIT (%)")
display(split_summary)

# 7. Validate the split
assert len(X_train) + len(X_test) == len(X)
assert X_train.index.intersection(X_test.index).empty
assert dates_sorted.loc[train_mask].max() < dates_sorted.loc[test_mask].min()
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

print("\nChronological split validation: PASSED")


In [ ]:

# ============================================================
# BASELINE MODEL: MAJORITY-CLASS PREDICTION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predict the majority class for every test observation
baseline_predictions = np.zeros(len(y_test), dtype=int)

# Calculate baseline metrics
baseline_accuracy = accuracy_score(
    y_test, baseline_predictions
)

baseline_precision = precision_score(
    y_test, baseline_predictions, zero_division=0
)

baseline_recall = recall_score(
    y_test, baseline_predictions, zero_division=0
)

baseline_f1 = f1_score(
    y_test, baseline_predictions, zero_division=0
)

# Display results
baseline_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],
    "Score": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1
    ]
})

baseline_results["Score"] = (
    baseline_results["Score"] * 100
).round(2)

display(baseline_results)

# Confusion matrix
baseline_cm = confusion_matrix(
    y_test, baseline_predictions
)

print("BASELINE CONFUSION MATRIX")
print(baseline_cm)

print("\nBASELINE CLASSIFICATION REPORT")
print(
    classification_report(
        y_test,
        baseline_predictions,
        target_names=[
            "Not Low Satisfaction",
            "Low Satisfaction"
        ],
        zero_division=0
    )
)


In [ ]:

# ============================================================
# INITIALISE MLFLOW USING SQLITE DATABASE
# ============================================================

import mlflow
from pathlib import Path

# Define the project directory
PROJECT_DIR = Path(
    r"C:\Users\HP\PycharmProjects\Learning\Nex_Project"
)

# Create the MLflow database path
MLFLOW_DB = PROJECT_DIR / "mlflow.db"

# Configure SQLite tracking backend
tracking_uri = f"sqlite:///{MLFLOW_DB.as_posix()}"

mlflow.set_tracking_uri(tracking_uri)

# Create or select the experiment
EXPERIMENT_NAME = "Olist_Low_Satisfaction_Prediction"

mlflow.set_experiment(EXPERIMENT_NAME)

# Retrieve experiment details
experiment = mlflow.get_experiment_by_name(
    EXPERIMENT_NAME
)

# Display tracking information
print("MLflow tracking initialised successfully")
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("Experiment status:", experiment.lifecycle_stage)
print("Database file:", MLFLOW_DB)


In [ ]:

# ============================================================
# LOGISTIC REGRESSION: TRAINING AND VALIDATION
# WITH MLFLOW EXPERIMENT TRACKING
# ============================================================

import mlflow
import mlflow.sklearn

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. Create a chronological validation split from training data
# ------------------------------------------------------------

train_dates = feature_df.loc[
    X_train.index, "order_purchase_timestamp"
]

sorted_train_indices = train_dates.sort_values().index

X_train_sorted = X_train.loc[sorted_train_indices].copy()
y_train_sorted = y_train.loc[sorted_train_indices].copy()
train_dates_sorted = train_dates.loc[sorted_train_indices].copy()

# Use the earliest 80% for model fitting
validation_position = int(len(X_train_sorted) * 0.80)
validation_cutoff = train_dates_sorted.iloc[validation_position]

fit_mask = train_dates_sorted < validation_cutoff
validation_mask = train_dates_sorted >= validation_cutoff

X_fit = X_train_sorted.loc[fit_mask].copy()
X_validation = X_train_sorted.loc[validation_mask].copy()

y_fit = y_train_sorted.loc[fit_mask].copy()
y_validation = y_train_sorted.loc[validation_mask].copy()

print("TRAINING AND VALIDATION SPLIT")
print("Model fitting rows:", len(X_fit))
print("Validation rows:", len(X_validation))
print("Validation cutoff:", validation_cutoff)

print("\nLow-satisfaction rate")
print("Model fitting:", round(y_fit.mean() * 100, 2), "%")
print("Validation:", round(y_validation.mean() * 100, 2), "%")

# ------------------------------------------------------------
# 2. Identify numerical and categorical features
# ------------------------------------------------------------

categorical_features = [
    "customer_state",
    "preferred_payment_type"
]

numerical_features = [
    "item_spending",
    "freight_spending",
    "total_items",
    "distinct_products",
    "category_diversity",
    "purchase_year",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour"
]

# ------------------------------------------------------------
# 3. Build preprocessing pipeline
# ------------------------------------------------------------

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# ------------------------------------------------------------
# 4. Build Logistic Regression pipeline
# ------------------------------------------------------------

logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

# ------------------------------------------------------------
# 5. Train and evaluate using the validation set
# ------------------------------------------------------------

with mlflow.start_run(run_name="Logistic_Regression_Baseline"):

    # Record experiment parameters
    mlflow.log_param("model", "Logistic Regression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 2000)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("validation_method", "Chronological")
    mlflow.log_param("validation_threshold", 0.5)

    # Fit only on the model-fitting period
    logistic_model.fit(X_fit, y_fit)

    # Generate validation predictions
    validation_predictions = logistic_model.predict(X_validation)

    validation_probabilities = (
        logistic_model.predict_proba(X_validation)[:, 1]
    )

    # Calculate validation metrics
    logistic_metrics = {
        "accuracy": accuracy_score(
            y_validation, validation_predictions
        ),
        "precision": precision_score(
            y_validation, validation_predictions, zero_division=0
        ),
        "recall": recall_score(
            y_validation, validation_predictions, zero_division=0
        ),
        "f1_score": f1_score(
            y_validation, validation_predictions, zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_validation, validation_probabilities
        ),
        "pr_auc": average_precision_score(
            y_validation, validation_probabilities
        )
    }

    # Log validation metrics in MLflow
    mlflow.log_metrics(logistic_metrics)

    # Log the fitted model.
    # numpy.dtype is explicitly trusted for skops serialization.
    mlflow.sklearn.log_model(
        sk_model=logistic_model,
        name="logistic_regression_model",
        skops_trusted_types=["numpy.dtype"]
    )

    # Capture the run ID before the run closes
    logistic_run_id = mlflow.active_run().info.run_id

# ------------------------------------------------------------
# 6. Display validation results
# ------------------------------------------------------------

print("\nLOGISTIC REGRESSION VALIDATION RESULTS")

logistic_results = pd.DataFrame({
    "Metric": list(logistic_metrics.keys()),
    "Score": list(logistic_metrics.values())
})

logistic_results["Score (%)"] = (
    logistic_results["Score"] * 100
).round(2)

display(logistic_results)

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_validation, validation_predictions))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_validation,
    validation_predictions,
    target_names=[
        "Not Low Satisfaction",
        "Low Satisfaction"
    ],
    zero_division=0
))

print("\nMLflow run ID:", logistic_run_id)


In [ ]:

# ============================================================
# CLASSIFICATION THRESHOLD ANALYSIS
# ============================================================

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Test different probability thresholds
thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]

threshold_results = []

for threshold in thresholds:

    # Convert predicted probabilities into class predictions
    threshold_predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    # Calculate classification metrics
    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y_validation, threshold_predictions
        ),
        "Precision": precision_score(
            y_validation, threshold_predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_validation, threshold_predictions,
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_validation, threshold_predictions,
            zero_division=0
        ),
        "Predicted Low-Satisfaction Orders": int(
            threshold_predictions.sum()
        )
    })

threshold_results_df = pd.DataFrame(threshold_results)

print("CLASSIFICATION THRESHOLD COMPARISON")

display(
    threshold_results_df.style.format({
        "Threshold": "{:.2f}",
        "Accuracy": "{:.2%}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1 Score": "{:.2%}"
    })
)

# Display confusion matrix for each threshold
for threshold in thresholds:

    threshold_predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    print(f"\nCONFUSION MATRIX — THRESHOLD {threshold:.2f}")
    print(confusion_matrix(
        y_validation,
        threshold_predictions
    ))


In [ ]:

# ============================================================
# FIND CANDIDATE CLASSIFICATION THRESHOLDS
# ============================================================



from sklearn.metrics import precision_recall_curve

# Calculate precision and recall across possible thresholds
precision_values, recall_values, threshold_values = (
    precision_recall_curve(
        y_validation,
        validation_probabilities
    )
)

# Calculate F1-score for each threshold
f1_values = (
    2 * precision_values[:-1] * recall_values[:-1]
    / (
        precision_values[:-1] + recall_values[:-1] + 1e-10
    )
)

# Identify the threshold with the highest F1-score
best_f1_index = np.argmax(f1_values)

best_f1_threshold = threshold_values[best_f1_index]

# Identify the best F1-score among thresholds with recall >= 50%
eligible_indices = np.where(recall_values[:-1] >= 0.50)[0]

if len(eligible_indices) > 0:
    best_recall_index = eligible_indices[
        np.argmax(f1_values[eligible_indices])
    ]

    best_recall_threshold = threshold_values[best_recall_index]
else:
    best_recall_threshold = None

# Display candidate thresholds
print("BEST F1-SCORE THRESHOLD")
print("Threshold:", round(best_f1_threshold, 4))
print("Precision:", round(precision_values[best_f1_index], 4))
print("Recall:", round(recall_values[best_f1_index], 4))
print("F1-score:", round(f1_values[best_f1_index], 4))

print("\nBEST F1-SCORE WITH AT LEAST 50% RECALL")

if best_recall_threshold is not None:
    print("Threshold:", round(best_recall_threshold, 4))
    print("Precision:", round(precision_values[best_recall_index], 4))
    print("Recall:", round(recall_values[best_recall_index], 4))
    print("F1-score:", round(f1_values[best_recall_index], 4))
else:
    print("No threshold achieved the required recall.")

# Compare the selected candidates
candidate_thresholds = [0.50, 0.60, 0.70, best_f1_threshold]

if best_recall_threshold is not None:
    candidate_thresholds.append(best_recall_threshold)

candidate_thresholds = sorted(set(candidate_thresholds))

comparison = []

for threshold in candidate_thresholds:
    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    comparison.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_validation, predictions, zero_division=0
        ),
        "Flagged Orders": int(predictions.sum())
    })

threshold_comparison = pd.DataFrame(comparison)

display(
    threshold_comparison.style.format({
        "Threshold": "{:.4f}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1 Score": "{:.2%}"
    })
)


In [ ]:

# ============================================================
# LOG THRESHOLD ANALYSIS IN MLFLOW
# ============================================================


with mlflow.start_run(run_name="Logistic_Regression_Threshold_Analysis"):

    mlflow.log_param("model", "Logistic Regression")
    mlflow.log_param("analysis", "Classification threshold selection")

    # Maximum F1-score threshold
    mlflow.log_param("best_f1_threshold", float(best_f1_threshold))

    mlflow.log_metric(
        "best_f1_precision",
        float(precision_values[best_f1_index])
    )
    mlflow.log_metric(
        "best_f1_recall",
        float(recall_values[best_f1_index])
    )
    mlflow.log_metric(
        "best_f1_score",
        float(f1_values[best_f1_index])
    )

    # Best F1-score threshold subject to at least 50% recall
    mlflow.log_param(
        "minimum_recall_threshold",
        float(best_recall_threshold)
    )

    mlflow.log_metric(
        "minimum_recall_precision",
        float(precision_values[best_recall_index])
    )
    mlflow.log_metric(
        "minimum_recall_actual_recall",
        float(recall_values[best_recall_index])
    )
    mlflow.log_metric(
        "minimum_recall_f1_score",
        float(f1_values[best_recall_index])
    )

    mlflow.log_param(
        "provisional_operating_threshold",
        float(best_recall_threshold)
    )

    threshold_run_id = mlflow.active_run().info.run_id

print("Threshold analysis recorded in MLflow.")
print("Threshold analysis run ID:", threshold_run_id)


In [ ]:

# ============================================================
# RANDOM FOREST: TRAINING AND VALIDATION
# WITH MLFLOW EXPERIMENT TRACKING
# ============================================================

import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. Build preprocessing pipeline
# ------------------------------------------------------------

# Numerical features do not require scaling for Random Forest.
numerical_pipeline_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Encode categorical features into numeric columns.
categorical_pipeline_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_rf = ColumnTransformer(transformers=[
    ("numerical", numerical_pipeline_rf, numerical_features),
    ("categorical", categorical_pipeline_rf, categorical_features)
])

# ------------------------------------------------------------
# 2. Build Random Forest pipeline
# ------------------------------------------------------------

random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# ------------------------------------------------------------
# 3. Train and evaluate using the same validation set
# ------------------------------------------------------------

with mlflow.start_run(run_name="Random_Forest_Baseline"):

    # Record model parameters
    mlflow.log_param("model", "Random Forest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 15)
    mlflow.log_param("min_samples_split", 10)
    mlflow.log_param("min_samples_leaf", 5)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("validation_method", "Chronological")

    # Fit only on the model-fitting period
    random_forest_model.fit(X_fit, y_fit)

    # Generate validation predictions
    rf_validation_predictions = (
        random_forest_model.predict(X_validation)
    )

    rf_validation_probabilities = (
        random_forest_model.predict_proba(X_validation)[:, 1]
    )

    # Calculate validation metrics
    rf_metrics = {
        "accuracy": accuracy_score(
            y_validation, rf_validation_predictions
        ),
        "precision": precision_score(
            y_validation, rf_validation_predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation, rf_validation_predictions,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_validation, rf_validation_predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_validation, rf_validation_probabilities
        ),
        "pr_auc": average_precision_score(
            y_validation, rf_validation_probabilities
        )
    }

    # Log validation metrics
    mlflow.log_metrics(rf_metrics)

    # Log fitted Random Forest model using skops serialization
    # These types are trusted because this is our own locally
    # trained model and its scikit-learn components.

    mlflow.sklearn.log_model(
        sk_model=random_forest_model,
        name="random_forest_model",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.tree._tree.Tree"
        ]
    )

    # Capture MLflow run ID
    rf_run_id = mlflow.active_run().info.run_id


# ------------------------------------------------------------
# 4. Display validation results
# ------------------------------------------------------------

print("\nRANDOM FOREST VALIDATION RESULTS")

rf_results = pd.DataFrame({
    "Metric": list(rf_metrics.keys()),
    "Score": list(rf_metrics.values())
})

rf_results["Score (%)"] = (
    rf_results["Score"] * 100
).round(2)

display(rf_results)

print("\nCONFUSION MATRIX")
print(confusion_matrix(
    y_validation,
    rf_validation_predictions
))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_validation,
    rf_validation_predictions,
    target_names=[
        "Not Low Satisfaction",
        "Low Satisfaction"
    ],
    zero_division=0
))

print("\nMLflow run ID:", rf_run_id)


In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np
import pandas as pd

# Obtain predicted probabilities for the positive class
rf_validation_probabilities = random_forest_model.predict_proba(
    X_validation
)[:, 1]

# Calculate precision and recall across possible thresholds
rf_precision, rf_recall, rf_thresholds = precision_recall_curve(
    y_validation,
    rf_validation_probabilities
)

# Calculate F1-score for each threshold
rf_f1_scores = (
    2 * rf_precision[:-1] * rf_recall[:-1]
    / (rf_precision[:-1] + rf_recall[:-1] + 1e-10)
)

# Find the threshold that produces the highest F1-score
best_f1_index = np.argmax(rf_f1_scores)

# Find the best F1-score while maintaining at least 50% recall
eligible_indices = np.where(rf_recall[:-1] >= 0.50)[0]

if len(eligible_indices) > 0:
    best_recall_index = eligible_indices[
        np.argmax(rf_f1_scores[eligible_indices])
    ]
else:
    best_recall_index = None

# Build threshold comparison results
threshold_results = [
    {
        "Selection": "Default threshold",
        "Threshold": 0.50
    },
    {
        "Selection": "Highest F1-score",
        "Threshold": rf_thresholds[best_f1_index]
    }
]

if best_recall_index is not None:
    threshold_results.append({
        "Selection": "Best F1 with recall >= 50%",
        "Threshold": rf_thresholds[best_recall_index]
    })

# Calculate the metrics for each selected threshold
results = []

for item in threshold_results:
    threshold = item["Threshold"]

    predictions = (
        rf_validation_probabilities >= threshold
    ).astype(int)

    from sklearn.metrics import (
        precision_score,
        recall_score,
        f1_score
    )

    results.append({
        "Selection": item["Selection"],
        "Threshold": round(threshold, 4),
        "Precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_validation, predictions, zero_division=0
        ),
        "Predicted positives": predictions.sum()
    })

rf_threshold_comparison = pd.DataFrame(results)

display(
    rf_threshold_comparison.style.format({
        "Threshold": "{:.4f}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1-score": "{:.2%}"
    })
)

In [ ]:
# Record the selected Random Forest threshold in MLflow

rf_selected_threshold = 0.5291

# Apply the selected threshold to validation probabilities
rf_selected_predictions = (
    rf_validation_probabilities >= rf_selected_threshold
).astype(int)

# Calculate metrics at the selected threshold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

rf_threshold_metrics = {
    "accuracy": accuracy_score(
        y_validation, rf_selected_predictions
    ),
    "precision": precision_score(
        y_validation, rf_selected_predictions, zero_division=0
    ),
    "recall": recall_score(
        y_validation, rf_selected_predictions, zero_division=0
    ),
    "f1_score": f1_score(
        y_validation, rf_selected_predictions, zero_division=0
    ),
    "predicted_positives": int(
        rf_selected_predictions.sum()
    )
}

with mlflow.start_run(run_name="Random_Forest_Threshold_Selection"):

    mlflow.log_param("model", "Random Forest")
    mlflow.log_param("selected_threshold", rf_selected_threshold)
    mlflow.log_param("threshold_selection_method",
                     "Best F1-score with recall >= 50%")
    mlflow.log_param("validation_method", "Chronological")

    mlflow.log_metrics(rf_threshold_metrics)

    rf_threshold_run_id = mlflow.active_run().info.run_id

print("Random Forest threshold analysis logged.")
print("Threshold:", rf_selected_threshold)
print("MLflow Run ID:", rf_threshold_run_id)

display(
    pd.DataFrame(
        list(rf_threshold_metrics.items()),
        columns=["Metric", "Score"]
    )
)

In [ ]:
# Inspect the steps in the trained Random Forest pipeline

print("Random Forest pipeline steps:")

for step_name, step_object in random_forest_model.named_steps.items():
    print(f"{step_name}: {type(step_object).__name__}")

In [ ]:
# ============================================================
# RANDOM FOREST HYPERPARAMETER TUNING
# Configure chronological cross-validation and search space
# ============================================================

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from scipy.stats import randint

# ------------------------------------------------------------
# 1. Define chronological cross-validation
# ------------------------------------------------------------
# The training data is already sorted chronologically.
# TimeSeriesSplit ensures each validation fold occurs after
# the observations used to train that fold.

time_cv = TimeSeriesSplit(n_splits=3)

# ------------------------------------------------------------
# 2. Define the Random Forest parameter search space
# ------------------------------------------------------------
# The "classifier__" prefix refers to the classifier step
# inside the existing pipeline.

rf_parameter_distributions = {
    "classifier__n_estimators": randint(100, 401),
    "classifier__max_depth": [10, 15, 20, 25, None],
    "classifier__min_samples_split": randint(5, 21),
    "classifier__min_samples_leaf": randint(2, 11),
    "classifier__max_features": ["sqrt", "log2"],
    "classifier__class_weight": [
        "balanced",
        "balanced_subsample"
    ]
}

# ------------------------------------------------------------
# 3. Configure randomized search
# ------------------------------------------------------------
# Average precision evaluates the model's precision-recall
# performance and is useful for imbalanced classification.

rf_random_search = RandomizedSearchCV(
    estimator=random_forest_model,
    param_distributions=rf_parameter_distributions,
    n_iter=20,
    scoring="average_precision",
    cv=time_cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True
)

print("Chronological cross-validation configured.")
print("Number of parameter combinations:", 20)
print("Number of CV folds:", 3)
print("Scoring metric: Average Precision")

In [ ]:
# ============================================================
# RANDOM FOREST HYPERPARAMETER TUNING
# Execute randomized search using chronological cross-validation
# ============================================================

import time

# ------------------------------------------------------------
# 1. Start the tuning process
# ------------------------------------------------------------

tuning_start_time = time.time()

print("Starting Random Forest hyperparameter tuning...")
print("This may take several minutes.")

rf_random_search.fit(X_fit, y_fit)

# ------------------------------------------------------------
# 2. Calculate tuning duration
# ------------------------------------------------------------

tuning_duration = time.time() - tuning_start_time

# ------------------------------------------------------------
# 3. Display the best hyperparameters and validation score
# ------------------------------------------------------------

print("\nHYPERPARAMETER TUNING COMPLETED")
print("-" * 50)

print("Best Average Precision:", round(
    rf_random_search.best_score_, 4
))

print("\nBest Hyperparameters:")

for parameter, value in rf_random_search.best_params_.items():
    print(f"{parameter}: {value}")

print("\nTuning duration:", round(tuning_duration / 60, 2), "minutes")

In [ ]:
# ============================================================
# TUNED RANDOM FOREST MODEL
# Evaluate the best model on the chronological validation set
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
import pandas as pd
import mlflow

# ------------------------------------------------------------
# 1. Retrieve the best model from randomized search
# ------------------------------------------------------------

tuned_random_forest = rf_random_search.best_estimator_

# ------------------------------------------------------------
# 2. Generate validation predictions at threshold 0.50
# ------------------------------------------------------------

tuned_rf_probabilities = tuned_random_forest.predict_proba(
    X_validation
)[:, 1]

tuned_rf_predictions = (
    tuned_rf_probabilities >= 0.50
).astype(int)

# ------------------------------------------------------------
# 3. Calculate validation performance metrics
# ------------------------------------------------------------

tuned_rf_metrics = {
    "accuracy": accuracy_score(
        y_validation, tuned_rf_predictions
    ),
    "precision": precision_score(
        y_validation, tuned_rf_predictions, zero_division=0
    ),
    "recall": recall_score(
        y_validation, tuned_rf_predictions, zero_division=0
    ),
    "f1_score": f1_score(
        y_validation, tuned_rf_predictions, zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_validation, tuned_rf_probabilities
    ),
    "pr_auc": average_precision_score(
        y_validation, tuned_rf_probabilities
    )
}

# ------------------------------------------------------------
# 4. Display evaluation results
# ------------------------------------------------------------

print("TUNED RANDOM FOREST VALIDATION RESULTS")
print("-" * 50)

display(
    pd.DataFrame(
        list(tuned_rf_metrics.items()),
        columns=["Metric", "Score"]
    )
)

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_validation, tuned_rf_predictions))

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_validation,
        tuned_rf_predictions,
        target_names=["Not Low Satisfaction", "Low Satisfaction"],
        zero_division=0
    )
)

# ------------------------------------------------------------
# 5. Record tuned model results in MLflow
# ------------------------------------------------------------

with mlflow.start_run(run_name="Tuned_Random_Forest_Validation"):

    mlflow.log_param("model", "Tuned Random Forest")
    mlflow.log_param("search_method", "RandomizedSearchCV")
    mlflow.log_param("cv_folds", 3)
    mlflow.log_param("scoring", "average_precision")
    mlflow.log_param("validation_threshold", 0.50)

    # Record the selected hyperparameters
    for parameter, value in rf_random_search.best_params_.items():
        mlflow.log_param(parameter, value)

    # Record cross-validation and validation performance
    mlflow.log_metric(
        "best_cv_average_precision",
        rf_random_search.best_score_
    )

    mlflow.log_metrics(tuned_rf_metrics)

    tuned_rf_run_id = mlflow.active_run().info.run_id

print("\nMLflow validation run ID:", tuned_rf_run_id)

In [ ]:
# ============================================================
# TUNED RANDOM FOREST
# Analyze decision thresholds using validation probabilities
# ============================================================

from sklearn.metrics import precision_recall_curve


# ------------------------------------------------------------
# 1. Calculate precision and recall at different thresholds
# ------------------------------------------------------------

tuned_precision, tuned_recall, tuned_thresholds = (
    precision_recall_curve(
        y_validation,
        tuned_rf_probabilities
    )
)

# ------------------------------------------------------------
# 2. Calculate F1-score for each threshold
# ------------------------------------------------------------

tuned_f1_scores = (
    2 * tuned_precision[:-1] * tuned_recall[:-1]
    / (
        tuned_precision[:-1]
        + tuned_recall[:-1]
        + 1e-10
    )
)

# ------------------------------------------------------------
# 3. Identify the threshold with the highest F1-score
# ------------------------------------------------------------

best_f1_index = np.argmax(tuned_f1_scores)

# ------------------------------------------------------------
# 4. Identify the best F1-score while maintaining at least
#    50% recall
# ------------------------------------------------------------

eligible_indices = np.where(
    tuned_recall[:-1] >= 0.50
)[0]

if len(eligible_indices) > 0:
    best_recall_index = eligible_indices[
        np.argmax(tuned_f1_scores[eligible_indices])
    ]
else:
    best_recall_index = None

# ------------------------------------------------------------
# 5. Display the selected threshold results
# ------------------------------------------------------------

threshold_results = [
    {
        "Selection": "Highest F1-score",
        "Threshold": tuned_thresholds[best_f1_index],
        "Precision": tuned_precision[best_f1_index],
        "Recall": tuned_recall[best_f1_index],
        "F1-score": tuned_f1_scores[best_f1_index]
    }
]

if best_recall_index is not None:
    threshold_results.append({
        "Selection": "Best F1 with recall >= 50%",
        "Threshold": tuned_thresholds[best_recall_index],
        "Precision": tuned_precision[best_recall_index],
        "Recall": tuned_recall[best_recall_index],
        "F1-score": tuned_f1_scores[best_recall_index]
    })

display(
    pd.DataFrame(threshold_results).style.format({
        "Threshold": "{:.4f}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1-score": "{:.2%}"
    })
)

In [ ]:
# ============================================================
# TUNED RANDOM FOREST
# Log selected classification threshold in MLflow
# ============================================================

import mlflow
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ------------------------------------------------------------
# 1. Define the selected threshold
# ------------------------------------------------------------

tuned_rf_selected_threshold = 0.5453

# ------------------------------------------------------------
# 2. Generate validation predictions
# ------------------------------------------------------------

tuned_rf_selected_predictions = (
    tuned_rf_probabilities >= tuned_rf_selected_threshold
).astype(int)

# ------------------------------------------------------------
# 3. Calculate validation metrics
# ------------------------------------------------------------

tuned_rf_threshold_metrics = {
    "accuracy": accuracy_score(
        y_validation, tuned_rf_selected_predictions
    ),
    "precision": precision_score(
        y_validation, tuned_rf_selected_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_validation, tuned_rf_selected_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_validation, tuned_rf_selected_predictions,
        zero_division=0
    )
}

# ------------------------------------------------------------
# 4. Record threshold and metrics in MLflow
# ------------------------------------------------------------

with mlflow.start_run(
    run_name="Tuned_RF_Threshold_Selection"
):

    mlflow.log_param(
        "model", "Tuned Random Forest"
    )

    mlflow.log_param(
        "selected_threshold",
        tuned_rf_selected_threshold
    )

    mlflow.log_param(
        "selection_method",
        "Highest F1-score with recall >= 50%"
    )

    mlflow.log_metrics(tuned_rf_threshold_metrics)

    tuned_rf_threshold_run_id = (
        mlflow.active_run().info.run_id
    )

print("Threshold analysis successfully logged.")
print("Selected threshold:", tuned_rf_selected_threshold)
print("MLflow Run ID:", tuned_rf_threshold_run_id)

display(
    pd.DataFrame(
        list(tuned_rf_threshold_metrics.items()),
        columns=["Metric", "Score"]
    )
)

In [ ]:
# ============================================================
# FINAL RANDOM FOREST MODEL
# Refit on complete training data and evaluate on test data
# ============================================================

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
import pandas as pd
import mlflow

# ------------------------------------------------------------
# 1. Create a fresh copy of the tuned pipeline
# ------------------------------------------------------------
# This uses the best hyperparameters found during tuning.
# The original tuned pipeline remains unchanged.

final_model = clone(rf_random_search.best_estimator_)

# ------------------------------------------------------------
# 2. Refit the model on the complete training dataset
# ------------------------------------------------------------

print("Training final Random Forest model...")

final_model.fit(X_train, y_train)

print("Final model training completed.")

# ------------------------------------------------------------
# 3. Generate probabilities for the untouched test dataset
# ------------------------------------------------------------

final_test_probabilities = final_model.predict_proba(
    X_test
)[:, 1]

# Apply the threshold selected using validation data
FINAL_THRESHOLD = 0.5453

final_test_predictions = (
    final_test_probabilities >= FINAL_THRESHOLD
).astype(int)

# ------------------------------------------------------------
# 4. Calculate final test metrics
# ------------------------------------------------------------

final_test_metrics = {
    "accuracy": accuracy_score(
        y_test, final_test_predictions
    ),
    "precision": precision_score(
        y_test, final_test_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test, final_test_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_test, final_test_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test, final_test_probabilities
    ),
    "pr_auc": average_precision_score(
        y_test, final_test_probabilities
    )
}

# ------------------------------------------------------------
# 5. Display the final test results
# ------------------------------------------------------------

print("\nFINAL RANDOM FOREST TEST RESULTS")
print("-" * 55)

display(
    pd.DataFrame(
        list(final_test_metrics.items()),
        columns=["Metric", "Score"]
    )
)

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, final_test_predictions))

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_test,
        final_test_predictions,
        target_names=[
            "Not Low Satisfaction",
            "Low Satisfaction"
        ],
        zero_division=0
    )
)

# ------------------------------------------------------------
# 6. Record final test results in MLflow
# ------------------------------------------------------------
# These metrics are the final holdout evaluation.
# Do not use this test result to adjust the threshold.

with mlflow.start_run(
    run_name="Final_Random_Forest_Test_Evaluation"
):

    mlflow.log_param(
        "model", "Tuned Random Forest"
    )

    mlflow.log_param(
        "selected_threshold", FINAL_THRESHOLD
    )

    mlflow.log_param(
        "evaluation_dataset", "Chronological holdout test set"
    )

    mlflow.log_metrics(final_test_metrics)

    final_test_run_id = mlflow.active_run().info.run_id

print("\nFinal test results logged in MLflow.")
print("MLflow Run ID:", final_test_run_id)

In [ ]:

# ============================================================
# FINAL RANDOM FOREST MODEL
# Visualize performance on the untouched test set
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

# Create three evaluation plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_test_predictions,
    display_labels=["Not Low Satisfaction", "Low Satisfaction"],
    cmap="Blues",
    ax=axes[0],
    colorbar=False
)
axes[0].set_title("Confusion Matrix")

# 2. ROC curve
RocCurveDisplay.from_predictions(
    y_test,
    final_test_probabilities,
    ax=axes[1]
)
axes[1].set_title("ROC Curve")

# 3. Precision-recall curve
PrecisionRecallDisplay.from_predictions(
    y_test,
    final_test_probabilities,
    ax=axes[2]
)
axes[2].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.show()

plt.savefig("final_model_evaluation.png", dpi=300, bbox_inches="tight")
plt.show()


In [3]:

# ============================================================
# SHAP EXPLAINABILITY
# Check whether the SHAP package is installed
# ============================================================

import importlib.util

if importlib.util.find_spec("shap") is not None:
    print("SHAP is already installed.")
else:
    print("SHAP is not installed.")
    print("Run this command in a separate notebook cell:")
    print("%pip install shap")


SHAP is not installed.
Run this command in a separate notebook cell:
%pip install shap


In [ ]:
%pip install shap

In [5]:

# ============================================================
# SHAP EXPLAINABILITY
# Calculate feature contributions for the Random Forest model
# ============================================================

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Select a manageable sample from the validation dataset
X_shap = X_validation.sample(
    n=min(300, len(X_validation)),
    random_state=42
)

# 2. Extract the preprocessing and fitted Random Forest components
preprocessor = final_model.named_steps["preprocessor"]
rf_classifier = final_model.named_steps["classifier"]

# 3. Apply the fitted preprocessing transformations
X_shap_transformed = preprocessor.transform(X_shap)

# Convert sparse output to a dense array for SHAP
if hasattr(X_shap_transformed, "toarray"):
    X_shap_transformed = X_shap_transformed.toarray()

# 4. Retrieve the transformed feature names
feature_names = preprocessor.get_feature_names_out()

# 5. Calculate SHAP values
explainer = shap.TreeExplainer(rf_classifier)
shap_values = explainer.shap_values(X_shap_transformed)

# 6. Select SHAP values for the positive class:
#    Class 1 represents Low Satisfaction
if isinstance(shap_values, list):
    positive_shap_values = shap_values[1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    positive_shap_values = shap_values[:, :, 1]
else:
    positive_shap_values = shap_values

print("SHAP calculation completed.")
print("Sample size:", len(X_shap))
print("Number of model features:", len(feature_names))
print("SHAP values shape:", positive_shap_values.shape)


C:\Users\HP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'X_validation' is not defined

In [6]:

# ============================================================
# SHAP GLOBAL FEATURE IMPORTANCE
# Identify the features with the greatest model influence
# ============================================================

# 1. Calculate mean absolute SHAP values
mean_abs_shap = np.abs(positive_shap_values).mean(axis=0)

# 2. Create a ranked feature-importance table
shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Absolute_SHAP": mean_abs_shap
})

shap_importance = shap_importance.sort_values(
    by="Mean_Absolute_SHAP",
    ascending=False
).reset_index(drop=True)

# 3. Display the 15 most influential features
print("TOP 15 SHAP FEATURES")
display(shap_importance.head(15).round(5))

# 4. Plot the top 15 features
top_features = shap_importance.head(15).sort_values(
    by="Mean_Absolute_SHAP",
    ascending=True
)

plt.figure(figsize=(10, 7))
plt.barh(
    top_features["Feature"],
    top_features["Mean_Absolute_SHAP"]
)

plt.title("Top 15 Features Influencing Low-Satisfaction Predictions")
plt.xlabel("Mean Absolute SHAP Value")
plt.ylabel("Model Feature")
plt.tight_layout()
plt.show()


NameError: name 'positive_shap_values' is not defined

In [ ]:

# ============================================================
# SHAP SUMMARY PLOT
# Examine the direction and distribution of feature effects
# ============================================================

# Convert transformed validation sample into a DataFrame
X_shap_display = pd.DataFrame(
    X_shap_transformed,
    columns=feature_names
)

# Generate SHAP summary plot
shap.summary_plot(
    positive_shap_values,
    X_shap_display,
    max_display=15,
    show=False
)

plt.title("SHAP Summary: Factors Influencing Low-Satisfaction Predictions")
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# SHAP LOCAL EXPLAINABILITY
# Select one correctly identified and one missed low-satisfaction order
# ============================================================

# 1. Generate predictions for the validation sample
validation_probabilities = final_model.predict_proba(X_shap)[:, 1]
validation_predictions = (
    validation_probabilities >= FINAL_THRESHOLD
).astype(int)

# Actual labels for the sampled validation orders
actual_labels = y_validation.loc[X_shap.index].to_numpy()

# 2. Identify correctly identified low-satisfaction orders
true_positive_indices = np.where(
    (actual_labels == 1) & (validation_predictions == 1)
)[0]

# 3. Identify missed low-satisfaction orders
false_negative_indices = np.where(
    (actual_labels == 1) & (validation_predictions == 0)
)[0]

print("Correctly identified low-satisfaction orders:", len(true_positive_indices))
print("Missed low-satisfaction orders:", len(false_negative_indices))

# 4. Select one example of each, if available
if len(true_positive_indices) > 0 and len(false_negative_indices) > 0:

    selected_indices = [
        true_positive_indices[0],
        false_negative_indices[0]
    ]

    for position, sample_index in enumerate(selected_indices):

        print("\nExample:",
              "Correctly identified" if position == 0 else "Missed order")
        print("Actual class:", actual_labels[sample_index])
        print("Predicted probability:", round(
            validation_probabilities[sample_index], 4
        ))
        print("Predicted class:", validation_predictions[sample_index])

else:
    print("One or both example types were not found in this sample.")


In [ ]:

# ============================================================
# SHAP LOCAL EXPLAINABILITY
# Compare one correctly identified and one missed order
# ============================================================

# Helper to obtain the SHAP base value for the positive class
expected_value = explainer.expected_value

if isinstance(expected_value, (list, np.ndarray)):
    base_value = expected_value[1]
else:
    base_value = expected_value

# Create a waterfall plot for each selected order
example_titles = [
    "Correctly Identified Low-Satisfaction Order",
    "Missed Low-Satisfaction Order"
]

for position, sample_index in enumerate(selected_indices):

    # Build a SHAP explanation for this individual order
    explanation = shap.Explanation(
        values=positive_shap_values[sample_index],
        base_values=base_value,
        data=X_shap_display.iloc[sample_index].values,
        feature_names=feature_names
    )

    print("\n" + example_titles[position])

    # Display the five most influential contributions
    shap.plots.waterfall(
        explanation,
        max_display=10,
        show=True
    )


In [ ]:

# ============================================================
# SHAP LOCAL EXPLAINABILITY
# Print the strongest feature contributions for two orders
# ============================================================

for position, sample_index in enumerate(selected_indices):

    print("\n" + "=" * 65)
    print(example_titles[position])
    print("=" * 65)

    # Get SHAP contributions for this order
    contributions = pd.DataFrame({
        "Feature": feature_names,
        "SHAP_Contribution": positive_shap_values[sample_index]
    })

    # Sort by absolute contribution
    contributions["Absolute_Contribution"] = (
        contributions["SHAP_Contribution"].abs()
    )

    contributions = contributions.sort_values(
        by="Absolute_Contribution",
        ascending=False
    )

    # Display the 10 strongest contributions
    display(
        contributions[
            ["Feature", "SHAP_Contribution"]
        ].head(10).round(5)
    )

    print("Base value:", round(float(base_value), 5))
    print(
        "Predicted probability:",
        round(float(validation_probabilities[sample_index]), 4)
    )


In [ ]:

# ============================================================
# FAIRNESS ANALYSIS
# Compare model performance across customer states
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    confusion_matrix
)

# 1. Generate predictions for the full validation set
fairness_probabilities = final_model.predict_proba(X_validation)[:, 1]

fairness_predictions = (
    fairness_probabilities >= FINAL_THRESHOLD
).astype(int)

fairness_actual = y_validation.loc[X_validation.index]

# 2. Attach actual outcomes and predictions to customer states
fairness_df = pd.DataFrame({
    "customer_state": X_validation["customer_state"].values,
    "actual": fairness_actual.values,
    "predicted": fairness_predictions
})

# 3. Calculate performance indicators for each state
state_results = []

for state, group in fairness_df.groupby("customer_state"):

    tn, fp, fn, tp = confusion_matrix(
        group["actual"],
        group["predicted"],
        labels=[0, 1]
    ).ravel()

    state_results.append({
        "State": state,
        "Orders": len(group),
        "Low_Satisfaction_Orders": int(group["actual"].sum()),
        "Positive_Prediction_Rate": group["predicted"].mean(),
        "Precision": precision_score(
            group["actual"],
            group["predicted"],
            zero_division=0
        ),
        "Recall_TPR": recall_score(
            group["actual"],
            group["predicted"],
            zero_division=0
        ),
        "False_Positive_Rate": (
            fp / (fp + tn) if (fp + tn) > 0 else np.nan
        )
    })

state_fairness = pd.DataFrame(state_results)

# 4. Retain states with at least 100 validation orders
state_fairness = state_fairness[
    state_fairness["Orders"] >= 100
].sort_values("Orders", ascending=False)

# 5. Display the geographic subgroup results
print("GEOGRAPHIC SUBGROUP PERFORMANCE")
display(state_fairness.round(4))

print("\nNumber of states included:", len(state_fairness))
print("Minimum group size: 100 validation orders")


In [ ]:

# ============================================================
# FAIRNESS ANALYSIS
# Calculate geographic disparity indicators
# ============================================================

# 1. Calculate the range of each fairness-related measure
metrics_to_compare = [
    "Positive_Prediction_Rate",
    "Recall_TPR",
    "False_Positive_Rate"
]

disparity_results = []

for metric in metrics_to_compare:

    minimum_value = state_fairness[metric].min()
    maximum_value = state_fairness[metric].max()

    disparity_results.append({
        "Metric": metric,
        "Minimum": minimum_value,
        "Maximum": maximum_value,
        "Absolute_Gap": maximum_value - minimum_value,
        "Maximum_to_Minimum_Ratio": (
            maximum_value / minimum_value
            if minimum_value > 0 else np.nan
        )
    })

disparity_summary = pd.DataFrame(disparity_results)

# 2. Display results
print("GEOGRAPHIC FAIRNESS DISPARITY SUMMARY")
display(disparity_summary.round(4))

# 3. Identify states with the highest and lowest positive prediction rates
print("\nPOSITIVE PREDICTION RATE COMPARISON")

highest_rate_state = state_fairness.loc[
    state_fairness["Positive_Prediction_Rate"].idxmax()
]

lowest_rate_state = state_fairness.loc[
    state_fairness["Positive_Prediction_Rate"].idxmin()
]

print(
    f"Highest: {highest_rate_state['State']} "
    f"({highest_rate_state['Positive_Prediction_Rate']:.2%})"
)

print(
    f"Lowest: {lowest_rate_state['State']} "
    f"({lowest_rate_state['Positive_Prediction_Rate']:.2%})"
)


In [ ]:

# ============================================================
# SENSITIVITY ANALYSIS
# Evaluate Random Forest performance at different thresholds
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 1. Define thresholds to compare
thresholds = [0.30, 0.40, 0.50, 0.5453, 0.60, 0.70, 0.80]

sensitivity_results = []

# 2. Evaluate the model at each threshold
for threshold in thresholds:

    # Convert predicted probabilities into class predictions
    threshold_predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    # Calculate confusion matrix values
    tn, fp, fn, tp = confusion_matrix(
        actual_labels,
        threshold_predictions,
        labels=[0, 1]
    ).ravel()

    # Calculate evaluation metrics
    sensitivity_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            actual_labels, threshold_predictions
        ),
        "Precision": precision_score(
            actual_labels, threshold_predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            actual_labels, threshold_predictions,
            zero_division=0
        ),
        "F1_Score": f1_score(
            actual_labels, threshold_predictions,
            zero_division=0
        ),
        "False_Positive_Rate": (
            fp / (fp + tn) if (fp + tn) > 0 else 0
        ),
        "Orders_Flagged": int(threshold_predictions.sum()),
        "Missed_Low_Satisfaction": int(fn)
    })

# 3. Display the sensitivity results
sensitivity_df = pd.DataFrame(sensitivity_results)

print("THRESHOLD SENSITIVITY ANALYSIS")
display(sensitivity_df.round(4))


In [ ]:

# ============================================================
# SENSITIVITY ANALYSIS
# Evaluate thresholds using the FULL validation dataset
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 1. Use the full validation probabilities and actual labels
full_validation_probabilities = fairness_probabilities
full_validation_actual = y_validation.loc[
    X_validation.index
].to_numpy()

# 2. Define thresholds for comparison
thresholds = [0.30, 0.40, 0.50, 0.5453, 0.60, 0.70, 0.80]

sensitivity_results = []

# 3. Calculate performance at each threshold
for threshold in thresholds:

    predictions = (
        full_validation_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        full_validation_actual,
        predictions,
        labels=[0, 1]
    ).ravel()

    sensitivity_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            full_validation_actual, predictions
        ),
        "Precision": precision_score(
            full_validation_actual, predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            full_validation_actual, predictions,
            zero_division=0
        ),
        "F1_Score": f1_score(
            full_validation_actual, predictions,
            zero_division=0
        ),
        "False_Positive_Rate": (
            fp / (fp + tn) if (fp + tn) > 0 else 0
        ),
        "Orders_Flagged": int(predictions.sum()),
        "Missed_Low_Satisfaction": int(fn)
    })

# 4. Display a clearly formatted table without row indices
sensitivity_df = pd.DataFrame(sensitivity_results)

print("FULL VALIDATION SET THRESHOLD SENSITIVITY")
print("Validation orders:", len(full_validation_actual))
print("Actual low-satisfaction orders:", int(full_validation_actual.sum()))

display(
    sensitivity_df.style.format({
        "Threshold": "{:.4f}",
        "Accuracy": "{:.2%}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1_Score": "{:.2%}",
        "False_Positive_Rate": "{:.2%}",
        "Orders_Flagged": "{:,.0f}",
        "Missed_Low_Satisfaction": "{:,.0f}"
    })
)


In [ ]:
# ============================================================
# SAVE THE FINAL TUNED RANDOM FOREST MODEL
# ============================================================

import joblib
from pathlib import Path

# Define the model output directory
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Define the model file path
MODEL_FILE = MODEL_DIR / "olist_low_satisfaction_model.joblib"

# Save the complete fitted pipeline
joblib.dump(final_model, MODEL_FILE)

# Confirm that the model file was saved
print("Model saved successfully.")
print("File location:", MODEL_FILE)
print("File size (MB):", round(MODEL_FILE.stat().st_size / (1024 ** 2), 2))

In [ ]:
# ============================================================
# VERIFY THE SAVED MODEL
# ============================================================

# Reload the saved model
loaded_model = joblib.load(MODEL_FILE)

# Generate probabilities using the saved model
saved_probabilities = loaded_model.predict_proba(X_test)[:, 1]

# Apply the same threshold used in the final evaluation
saved_predictions = (
    saved_probabilities >= FINAL_THRESHOLD
).astype(int)

# Compare predictions and probabilities with the original model
predictions_match = np.array_equal(
    saved_predictions,
    final_test_predictions
)

probabilities_match = np.allclose(
    saved_probabilities,
    final_test_probabilities
)

print("Saved model loaded successfully.")
print("Test predictions generated:", len(saved_predictions))
print("Prediction probabilities generated:", len(saved_probabilities))
print("Classification threshold:", FINAL_THRESHOLD)
print("Predictions match original model:", predictions_match)
print("Probabilities match original model:", probabilities_match)

# Confirm that the probabilities are valid
print(
    "Probability range:",
    round(saved_probabilities.min(), 4),
    "to",
    round(saved_probabilities.max(), 4)
)

# Verification checks
assert predictions_match, "Predictions do not match."
assert probabilities_match, "Probabilities do not match."
assert np.isfinite(saved_probabilities).all()

print("\nModel verification: PASSED")

In [5]:
# ============================================================
# SAVE MODEL CONFIGURATION
# ============================================================

import json

# Define the configuration file
CONFIG_FILE = MODEL_DIR / "model_config.json"

# Store the model's prediction threshold and target definition
model_config = {
    "model_name": "Olist Low Satisfaction Prediction",
    "model_type": "Tuned Random Forest",
    "classification_threshold": FINAL_THRESHOLD,
    "positive_class": 1,
    "positive_class_definition": "Average review score <= 2",
    "prediction_unit": "Order"
}

# Save configuration as JSON
with open(CONFIG_FILE, "w", encoding="utf-8") as file:
    json.dump(model_config, file, indent=4)

print("Model configuration saved successfully.")
print("Configuration file:", CONFIG_FILE)
print(json.dumps(model_config, indent=4))

NameError: name 'MODEL_DIR' is not defined

In [ ]:
# ============================================================
# STEP 4: INSTALL FASTAPI DEPENDENCIES
# ============================================================

%pip install fastapi uvicorn

In [ ]:
# ============================================================
# INSPECT THE MODEL INPUT FEATURES
# ============================================================

# Display the features expected by the fitted pipeline
expected_features = loaded_model.feature_names_in_

print("Number of expected input features:", len(expected_features))
print("\nExpected API input features:")

for position, feature in enumerate(expected_features, start=1):
    print(f"{position}. {feature}")

In [ ]:
# ============================================================
# CREATE THE FASTAPI APPLICATION
# ============================================================

api_code = '''
# ============================================================
# OLIST LOW CUSTOMER SATISFACTION PREDICTION API
# ============================================================

from pathlib import Path
import json
import joblib

from fastapi import FastAPI
from pydantic import BaseModel, Field


# ------------------------------------------------------------
# 1. LOAD THE SAVED MODEL AND CONFIGURATION
# ------------------------------------------------------------

PROJECT_DIR = Path(__file__).resolve().parent
MODEL_DIR = PROJECT_DIR / "models"

MODEL_FILE = MODEL_DIR / "olist_low_satisfaction_model.joblib"
CONFIG_FILE = MODEL_DIR / "model_config.json"

model = joblib.load(MODEL_FILE)

with open(CONFIG_FILE, "r", encoding="utf-8") as file:
    config = json.load(file)

THRESHOLD = config["classification_threshold"]


# ------------------------------------------------------------
# 2. INITIALISE FASTAPI
# ------------------------------------------------------------

app = FastAPI(
    title="Olist Customer Satisfaction Prediction API",
    description="Predicts the likelihood of low customer satisfaction for an order.",
    version="1.0.0"
)


# ------------------------------------------------------------
# 3. DEFINE THE INPUT DATA STRUCTURE
# ------------------------------------------------------------

class OrderFeatures(BaseModel):

    customer_state: str

    item_spending: float = Field(ge=0)
    freight_spending: float = Field(ge=0)

    total_items: float = Field(ge=0)
    distinct_products: float = Field(ge=0)
    category_diversity: float = Field(ge=0)

    preferred_payment_type: str

    purchase_year: int
    purchase_month: int = Field(ge=1, le=12)
    purchase_day_of_week: int = Field(ge=0, le=6)
    purchase_hour: int = Field(ge=0, le=23)


# ------------------------------------------------------------
# 4. CREATE A BASIC HEALTH-CHECK ENDPOINT
# ------------------------------------------------------------

@app.get("/")
def home():
    return {
        "message": "Olist Low Satisfaction Prediction API is running",
        "model": config["model_name"],
        "threshold": THRESHOLD
    }


# ------------------------------------------------------------
# 5. CREATE THE PREDICTION ENDPOINT
# ------------------------------------------------------------

@app.post("/predict")
def predict_order(features: OrderFeatures):

    input_data = features.model_dump()

    probability = float(
        model.predict_proba(
            __import__("pandas").DataFrame([input_data])
        )[0, 1]
    )

    prediction = int(probability >= THRESHOLD)

    return {
        "low_satisfaction_prediction": prediction,
        "low_satisfaction_probability": round(probability, 4),
        "classification_threshold": THRESHOLD,
        "risk_category": (
            "High risk" if prediction == 1 else "Lower risk"
        ),
        "interpretation": (
            "The order is predicted to have low customer satisfaction."
            if prediction == 1
            else "The order is not predicted to have low customer satisfaction."
        )
    }
'''

# Save the API source code
API_FILE = PROJECT_DIR / "api.py"

with open(API_FILE, "w", encoding="utf-8") as file:
    file.write(api_code)

print("FastAPI application created successfully.")
print("API file:", API_FILE)

In [ ]:
# ============================================================
# STEP 7A: VERIFY THE PROJECT DIRECTORY
# ============================================================

print("Project directory exists:", PROJECT_DIR.exists())
print("Project directory:", PROJECT_DIR)
print("API file exists:", API_FILE.exists())
print("API file:", API_FILE)

In [ ]:
# ============================================================
# REVIEW MLFLOW EXPERIMENT RUNS
# ============================================================

import mlflow

# Connect to the existing MLflow tracking database
mlflow.set_tracking_uri(
    f"sqlite:///{(PROJECT_DIR / 'mlflow.db').as_posix()}"
)

# Retrieve the Olist experiment
experiment = mlflow.get_experiment_by_name(
    "Olist_Low_Satisfaction_Prediction"
)

# Retrieve all recorded runs
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"]
)

# Display experiment information
print("Experiment:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("Total recorded runs:", len(runs))

# Display the main tracking information
display_columns = [
    column for column in [
        "run_id",
        "status",
        "tags.mlflow.runName",
        "metrics.accuracy",
        "metrics.precision",
        "metrics.recall",
        "metrics.f1",
        "metrics.roc_auc",
        "metrics.pr_auc"
    ]
    if column in runs.columns
]

display(runs[display_columns])

In [ ]:
# ============================================================
# INSPECT KEY MLFLOW RUNS AND ARTIFACTS
# ============================================================

from mlflow.tracking import MlflowClient

# Connect to the existing MLflow tracking database
client = MlflowClient()

# Identify the two most important runs
key_runs = {
    "Tuned Random Forest Validation": "28627614598341d79e410cffa8f67f14",
    "Final Random Forest Test Evaluation": "b7ea5460d9bf423d98c99203e5ba4ddc"
}

# Inspect each run
for run_name, run_id in key_runs.items():

    run = client.get_run(run_id)

    print("\n" + "=" * 65)
    print(run_name)
    print("=" * 65)

    print("Run ID:", run.info.run_id)
    print("Status:", run.info.status)

    print("\nParameters:")
    for parameter, value in run.data.params.items():
        print(f"{parameter}: {value}")

    print("\nMetrics:")
    for metric, value in run.data.metrics.items():
        print(f"{metric}: {value}")

    print("\nSaved artifacts:")
    artifacts = client.list_artifacts(run_id)

    if artifacts:
        for artifact in artifacts:
            print(artifact.path)
    else:
        print("No artifacts recorded for this run.")

In [ ]:
# ============================================================
# SAVE FINAL EVALUATION ARTIFACTS TO MLFLOW
# ============================================================

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

# Create a temporary folder for evaluation artifacts
ARTIFACT_DIR = MODEL_DIR / "mlflow_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. GENERATE THE CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(y_test, final_test_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not Low Satisfaction", "Low Satisfaction"],
    yticklabels=["Not Low Satisfaction", "Low Satisfaction"]
)

plt.title("Final Random Forest Test Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()

confusion_matrix_file = ARTIFACT_DIR / "confusion_matrix.png"
plt.savefig(confusion_matrix_file, dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 2. SAVE THE CLASSIFICATION REPORT
# ------------------------------------------------------------

report = classification_report(
    y_test,
    final_test_predictions,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

classification_file = ARTIFACT_DIR / "classification_report.csv"
report_df.to_csv(classification_file)

# ------------------------------------------------------------
# 3. SAVE THE FINAL TEST METRICS
# ------------------------------------------------------------

test_metrics = {
    "accuracy": 0.8641936661971096,
    "precision": 0.24610169491525424,
    "recall": 0.19579288025889968,
    "f1_score": 0.21808350856112946,
    "roc_auc": 0.5817478444809947,
    "pr_auc": 0.1697818287038106,
    "classification_threshold": FINAL_THRESHOLD
}

metrics_file = ARTIFACT_DIR / "final_test_metrics.json"

with open(metrics_file, "w", encoding="utf-8") as file:
    json.dump(test_metrics, file, indent=4)

# ------------------------------------------------------------
# 4. ATTACH THE FILES TO THE FINAL MLFLOW RUN
# ------------------------------------------------------------

FINAL_RUN_ID = "b7ea5460d9bf423d98c99203e5ba4ddc"

with mlflow.start_run(run_id=FINAL_RUN_ID):
    mlflow.log_artifacts(str(ARTIFACT_DIR))

print("Evaluation artifacts saved to MLflow successfully.")
print("Files attached:")
print("- Confusion matrix")
print("- Classification report")
print("- Final test metrics")

In [ ]:
# ============================================================
# VERIFY MLFLOW EVALUATION ARTIFACTS
# ============================================================

from mlflow.tracking import MlflowClient

# Connect to MLflow
client = MlflowClient()

# Retrieve artifacts from the final test evaluation run
FINAL_RUN_ID = "b7ea5460d9bf423d98c99203e5ba4ddc"

artifacts = client.list_artifacts(FINAL_RUN_ID)

print("Final evaluation run:", FINAL_RUN_ID)
print("\nAttached artifacts:")

for artifact in artifacts:
    print("-", artifact.path)

# Verify the expected files
artifact_names = {artifact.path for artifact in artifacts}

expected_artifacts = {
    "confusion_matrix.png",
    "classification_report.csv",
    "final_test_metrics.json"
}

missing_artifacts = expected_artifacts - artifact_names

assert not missing_artifacts, (
    f"Missing artifacts: {missing_artifacts}"
)

print("\nArtifact verification: PASSED")